In [ ]:
#General imports
resol = 300
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
})

import sys
import numpy as np
from pathlib import Path
pi = np.pi

project_root = Path().resolve().parents[0]
sys.path.append(str(project_root))

#Local imports
from experiments.coincidence_vs_n import run_coincidence_vs_n

<h2> Execute the experiment </h2>

Given an off-resonant scattering experiment, I check the convergence of the coincidence against $\Lambda_{\rm UV}$ for $\Lambda_{\rm IR} = 0$ for different values of $n$.

In [ ]:
#Prepare the photon frequencies
omega_q_tab = [9*pi, 7*pi, 5*pi]
ir_val = 2*pi

n_bare_tab = np.arange(5)

nb_freq_window = 10
ir_tab = ir_val + np.zeros(nb_freq_window)
uv_tab = np.linspace(16*pi, 36*pi, nb_freq_window)

#for q in range(len(omega_q_tab)):
for q in [0]:

    index_omega_q = q + 1
    omega_q = omega_q_tab[q]
    
    for n in n_bare_tab:
        print("Running n = ", n, " out of ", n_bare_tab[-1])
        _, _, coincidence_tab = run_coincidence_vs_n(omega_q, ir_tab, uv_tab, index_omega_q, n, 
                                                     store_results=True, progress=True)
        
        print("------------- \n")

<h1> Results </h1>

In [ ]:
omega_A = 10*pi
Gamma = 5*pi
delta_q = 0.05*pi

omega_q_tab = [9*pi, 7*pi, 5*pi]
ir_val = 0*pi
index_omega_q = 1

#Theoretical value
R_theory_physical = 1 / (1 + ((omega_q_tab[index_omega_q-1] - omega_A)/ (Gamma/2))**2)
theoretical_val = 1 - 4*R_theory_physical*(1-R_theory_physical)

if np.abs(theoretical_val) < 1e-1: #Non-monochromatic correction
    non_monochr_ratio = Gamma / (2*delta_q)
    theoretical_val += 1/(np.sqrt(pi) * non_monochr_ratio)

n_bare_tab = np.arange(5)
coincidence_to_plot = []
relative_errors_to_plot = []
lbda_tab_to_plot = []

for i in range(len(n_bare_tab)):
    #Recover the data
    data_file = f"../results/csv_files/coincidence_vs_n{i}_ir{int(ir_tab[0]/pi)}_{index_omega_q}.csv"

    df = pd.read_csv(data_file)
    coincidence_to_plot.append(df['coincidence_tab'].to_numpy())
    ir_tab = df['ir_tab'].to_numpy()
    uv_tab = df['uv_tab'].to_numpy()
    lbda_tab_to_plot.append(0.5*(uv_tab - ir_tab))

    #Comparision with theoretical value for each scattering experiment
    relative_errors_to_plot.append(np.abs(coincidence_to_plot[-1] - theoretical_val) / theoretical_val)

Create the figure

In [ ]:
# Figure setup (même ADN que ta figure de référence)
fig, ax = plt.subplots(figsize=(4, 3), dpi=300)

# Palette sobre + markers distincts
colors  = ["#ad7100", "#b74744", "#903d6e", "#4b4477", "#003f5c"]
labels  = [r'Baseline', r'$n = 1$', r'$n = 2$', r'$n = 3$', r'$n = 4$']
lbda_conv = np.zeros(len(n_bare_tab))

# Scatter plots
for i in range(len(n_bare_tab)):
    #Index where the curve enters the 5% confidence region
    if (relative_errors_to_plot[i][-1] < 0.05):
        lbda_conv[i] = lbda_tab_to_plot[i][np.where(relative_errors_to_plot[i] < 0.05)[0][0]]
        labels[i] += '(O)'
    else:
        lbda_conv[i] = np.inf
        labels[i] += '(X)'

    print(f"Relative error for n = {n_bare_tab[i]:.1f} : {relative_errors_to_plot[i][-1]*100:.3f} %, reached at Lbda =  {lbda_conv[i] / pi :.3f} pi")

    ax.plot(
        lbda_tab_to_plot[i],
        coincidence_to_plot[i],
        marker="o",
        color=colors[i],
        label = labels[i],
        markersize=1,
        linewidth=0.6,
        alpha=0.9,
        zorder=3
    )

#5% confidence region
ax.hlines(theoretical_val, 0, 100*pi,color='green', alpha=0.5, linewidth=0.8, linestyle='--')
ax.fill_between(np.linspace(0, 100*pi, 100),
                0.95*theoretical_val,
                1.05*theoretical_val,color=colors[i],alpha=0.12,linewidth=0, zorder=1)


#legend outside the plot
ax.legend(
    prop={'size': 8},
    frameon=True,
    loc='center left',
    bbox_to_anchor=(1.02, 0.5)
)

#Switch the x-axis in pi units
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2*pi)) 
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/pi:.0f}$\\pi$'))
ax.set_xlim([lbda_tab_to_plot[0][0] - 1, lbda_tab_to_plot[0][-1] + 1])

# Labels
ax.set_xlabel(r'\textbf{Bandwidth} $\Lambda$', fontsize=13)
ax.set_ylabel(r'\textbf{Coincidence} $\mathcal{C}$', fontsize=13)

#grid
ax.grid(color='0.9', linestyle='-', linewidth=0.4)


# Tick appearance
ax.tick_params(axis='both', which='major', length=5, width=0.8)
ax.tick_params(axis='both', which='minor', length=3, width=0.6)

#font size
for item in [ax.xaxis.label, ax.yaxis.label]:
    item.set_fontsize(10)

for item in (ax.get_xticklabels() + ax.get_yticklabels()):
    item.set_fontsize(10)

plt.tight_layout()
plt.savefig(f'../results/fig/coincidence_vs_n{n}_ir{int(ir_tab[0]/pi)}_{index_omega_q}.pdf')
plt.show()